# BioRAG-X — 05 Agentic Chunking

## LLM-guided Chunking + Agentic Chunking + Adaptive Chunking Router

This notebook adds the **decision layer** after the chunking strategies from Notebooks 03–04.

We distinguish:

**Adaptive chunking**
```text
Document → profile → policy → choose chunker
```

**Agentic chunking**
```text
Document → inspect → choose tool → execute → evaluate → refine/stop
```

**LLM-guided chunking**
```text
Document + explicit rules → LLM proposes boundaries/metadata → validator checks them
```

Production principle:

> The fast adaptive router is the default path; expensive LLM-guided/agentic chunking is used only when confidence is low or the document is unusually complex.

Research references:
- Adaptive Chunking: https://arxiv.org/abs/2603.25333
- TopoChunker: https://arxiv.org/abs/2603.18409
- PIC / Document Segmentation Matters: https://aclanthology.org/2025.findings-acl.422/

In [1]:
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter
import hashlib, json, re, os, time
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import AzureOpenAI

SEED = 42
CANONICAL_DIR = Path("data/canonical")
ARTIFACT_DIR = Path("artifacts/05_agentic_chunking")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PASSAGES_PATH = CANONICAL_DIR / "passages.parquet"
if not PASSAGES_PATH.exists():
    raise FileNotFoundError("Run Notebook 02 first.")

passages = pd.read_parquet(PASSAGES_PATH)
passages["retrieval_text"] = passages["retrieval_text"].fillna("").astype(str)

TOKEN_RE = re.compile(r"\S+")

def tokens(text): return TOKEN_RE.findall(text or "")
def n_tokens(text): return len(tokens(text))

def sentences(text):
    text = (text or "").strip()
    if not text: return []
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text) if s.strip()]

print("Passages:", len(passages))

Passages: 40221


## 0b. Azure OpenAI chat client (for real LLM-guided chunking)

LLM-guided chunking is a **real** capability here, not a stub. We instantiate an
Azure OpenAI chat client and probe it once. If the probe fails, `llm_available`
is set to `False` and every LLM path falls back to the deterministic router — so
the notebook still runs end-to-end without a model, and says so honestly.

Consistent with the project design, the LLM is a **proposal engine** whose output
is always checked by `validate_llm_plan` before any chunk is created.


In [2]:
# Real Azure OpenAI chat client for LLM-guided chunking.
ENV_PATH = Path("..") / ".env"
load_dotenv(ENV_PATH if ENV_PATH.exists() else None)

CHAT_DEPLOYMENT = os.getenv("AZURE_DEPLOYMENT_GPT_4o", "gpt-4o")
LLM_MODEL_NAME = f"azure-openai:{CHAT_DEPLOYMENT}"

chat_client = None
llm_available = False
llm_usage = {"calls": 0, "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0, "seconds": 0.0}

try:
    chat_client = AzureOpenAI(
        api_key=os.getenv("AZURE_GPT_4o_API_KEY"),
        api_version=os.getenv("AZURE_GPT_4o_API_VERSION"),
        azure_endpoint=os.getenv("AZURE_GPT_4o_ENDPOINT"),
    )
    _probe = chat_client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[{"role": "user", "content": "Reply with the single word: OK"}],
        max_tokens=5, temperature=0,
    )
    llm_available = _probe.choices[0].message.content.strip().upper().startswith("OK")
except Exception as e:
    print("Azure chat client unavailable; LLM-guided chunking will fall back to deterministic. Reason:",
          type(e).__name__, str(e)[:160])

def _chat_json(prompt, max_tokens=700, temperature=0.0):
    """Call the chat model and return (parsed_json_or_None, raw_text). Records usage."""
    if chat_client is None:
        return None, ""
    t0 = time.perf_counter()
    resp = chat_client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": "You are a precise JSON-only API. Output only valid JSON."},
            {"role": "user", "content": prompt},
        ],
        max_tokens=max_tokens, temperature=temperature,
        response_format={"type": "json_object"},
    )
    llm_usage["seconds"] += time.perf_counter() - t0
    llm_usage["calls"] += 1
    if resp.usage:
        llm_usage["prompt_tokens"] += resp.usage.prompt_tokens
        llm_usage["completion_tokens"] += resp.usage.completion_tokens
        llm_usage["total_tokens"] += resp.usage.total_tokens
    raw = resp.choices[0].message.content or ""
    try:
        return json.loads(raw), raw
    except Exception:
        # Best-effort: extract the outermost JSON object.
        m = re.search(r"\{.*\}", raw, re.S)
        if m:
            try:
                return json.loads(m.group(0)), raw
            except Exception:
                return None, raw
        return None, raw

print("Chat backend:", LLM_MODEL_NAME, "| llm_available:", llm_available)


Chat backend: azure-openai:gpt-4o | llm_available: True


## 1. LLM-guided chunking

The LLM is a **proposal engine**, not the source of truth.

It may propose:
- sentence boundaries
- topic boundaries
- labels/reasons
- preservation constraints

The validator must reject:
- invalid boundary indexes
- malformed output
- impossible sizes
- non-reconstructable text
- unsupported structure

We use a closed JSON contract.

In [3]:
def build_llm_chunking_prompt(text, max_tokens=384):
    return f'''
You are the BioRAG-X biomedical document segmentation engine.

Return JSON only:
{{
  "boundaries": [sentence_index_after_boundary],
  "labels": ["short reason"],
  "constraints": {{
    "preserve_entities": true,
    "preserve_relations": true,
    "preserve_negation": true
  }}
}}

Rules:
1. Preserve the original wording.
2. Prefer sentence boundaries.
3. Do not split biomedical entities or relation statements.
4. Preserve negation with the statement it negates.
5. Prefer chunks <= {max_tokens} approximate tokens.

TEXT:
{text}
'''

def validate_llm_plan(plan, sentence_count, max_boundaries=None):
    errors = []
    if not isinstance(plan, dict):
        return False, ["Plan is not an object."]

    boundaries = plan.get("boundaries")
    if not isinstance(boundaries, list):
        errors.append("boundaries must be a list")
        boundaries = []

    if any(not isinstance(x, int) for x in boundaries):
        errors.append("boundaries must contain integers")

    if any(x < 1 or x >= sentence_count for x in boundaries):
        errors.append("boundary outside valid range")

    if boundaries != sorted(set(boundaries)):
        errors.append("boundaries must be sorted and unique")

    if max_boundaries is not None and len(boundaries) > max_boundaries:
        errors.append("too many boundaries")

    return len(errors) == 0, errors

demo = "BRCA1 participates in DNA repair. Loss of BRCA1 causes homologous recombination deficiency. PARP inhibitors can exploit this deficiency."
print(build_llm_chunking_prompt(demo, 64))
print(validate_llm_plan({"boundaries":[2]}, len(sentences(demo))))

def apply_boundaries(sents, boundaries):
    """Turn sentence-index boundaries into grouped text chunks."""
    cuts = [0] + sorted(set(int(b) for b in boundaries)) + [len(sents)]
    out = []
    for a, b in zip(cuts[:-1], cuts[1:]):
        piece = " ".join(sents[a:b]).strip()
        if piece:
            out.append(piece)
    return out

def llm_guided_chunks(text, max_tokens=256, max_repair=1):
    """REAL LLM-guided chunking.

    Returns (chunks, meta). meta records whether the LLM was actually used, the
    validation outcome, and the fallback reason if any. Falls back to the
    deterministic recursive chunker on unavailable model / invalid output.
    """
    sents = sentences(text)
    meta = {"llm_used": False, "valid": False, "fallback": None, "boundaries": None, "labels": None}
    if len(sents) < 2:
        meta["fallback"] = "too_few_sentences"
        return recursive_chunks(text, max_tokens), meta
    if not llm_available:
        meta["fallback"] = "llm_unavailable"
        return recursive_chunks(text, max_tokens), meta

    prompt = build_llm_chunking_prompt(text, max_tokens)
    for attempt in range(max_repair + 1):
        plan, raw = _chat_json(prompt)
        meta["llm_used"] = True
        if plan is None:
            prompt = build_llm_chunking_prompt(text, max_tokens) + "\n\nReturn ONLY valid JSON."
            continue
        ok, errors = validate_llm_plan(plan, len(sents), max_boundaries=max(1, len(sents) - 1))
        if ok:
            meta["valid"] = True
            meta["boundaries"] = list(plan.get("boundaries", []))
            meta["labels"] = plan.get("labels")
            chunks = apply_boundaries(sents, meta["boundaries"])
            return (chunks if chunks else recursive_chunks(text, max_tokens)), meta
        # repair attempt: tell the model what was wrong
        prompt = (build_llm_chunking_prompt(text, max_tokens)
                  + f"\n\nYour previous output was invalid: {errors}. "
                    f"There are {len(sents)} sentences (valid boundary indexes 1..{len(sents)-1}).")

    meta["fallback"] = "invalid_after_repair"
    return recursive_chunks(text, max_tokens), meta



You are the BioRAG-X biomedical document segmentation engine.

Return JSON only:
{
  "boundaries": [sentence_index_after_boundary],
  "labels": ["short reason"],
  "constraints": {
    "preserve_entities": true,
    "preserve_relations": true,
    "preserve_negation": true
  }
}

Rules:
1. Preserve the original wording.
2. Prefer sentence boundaries.
3. Do not split biomedical entities or relation statements.
4. Preserve negation with the statement it negates.
5. Prefer chunks <= 64 approximate tokens.

TEXT:
BRCA1 participates in DNA repair. Loss of BRCA1 causes homologous recombination deficiency. PARP inhibitors can exploit this deficiency.

(True, [])


## 2. Chunking toolbox

The agent must call **known tools**, not generate arbitrary code.

Tools exposed here:
- fixed
- recursive
- semantic
- biomedical
- proposition
- parent_child
- late

Notebook 04 contains richer implementations; here we keep a lightweight toolbox so the routing logic is independently understandable.

In [4]:
BIO = {
    "entity": re.compile(r"\b(?:BRCA\d*|EGFR|KRAS|BRAF|p53|COX[- ]?2|aspirin|olaparib|breast cancer|ovarian cancer)\b", re.I),
    "relation": re.compile(r"\b(?:inhibits?|activates?|associated with|increases?|decreases?|causes?|reduces?|induces?|interacts with)\b", re.I),
    "negation": re.compile(r"\b(?:no|not|never|without|absence of|did not|does not)\b", re.I),
}

def fixed_chunks(text, size=256, overlap=32):
    x = tokens(text)
    step = size - overlap
    out, start = [], 0
    while start < len(x):
        end = min(start + size, len(x))
        out.append(" ".join(x[start:end]))
        if end == len(x): break
        start += step
    return out

def recursive_chunks(text, max_tokens=256):
    groups, cur = [], []
    for s in sentences(text):
        if cur and n_tokens(" ".join(cur + [s])) > max_tokens:
            groups.append(" ".join(cur))
            cur = [s]
        else:
            cur.append(s)
    if cur: groups.append(" ".join(cur))
    return groups

def proposition_chunks(text):
    out = []
    for s in sentences(text):
        parts = re.split(r"\s+(?:and|but|while|whereas|although|however)\s+", s, flags=re.I)
        out.extend(p.strip(" ,;:") for p in parts if p.strip())
    return out

def parent_child_chunks(text, child_size=96, overlap=16):
    return fixed_chunks(text, child_size, overlap)

def late_spans(text, size=256, overlap=32):
    return fixed_chunks(text, size, overlap)

def semantic_chunks(text, max_tokens=256):
    # Lexical-shift boundary detection (no embedding dependency in NB05):
    # start a new chunk when adjacent-sentence token overlap is low OR size exceeds budget.
    sents = sentences(text)
    if not sents:
        return []
    groups, cur = [], [sents[0]]
    prev_set = set(re.findall(r"[A-Za-z0-9-]+", sents[0].lower()))
    for s in sents[1:]:
        cur_set = set(re.findall(r"[A-Za-z0-9-]+", s.lower()))
        overlap = len(prev_set & cur_set) / max(1, len(prev_set | cur_set))
        shift = overlap < 0.15
        too_big = n_tokens(" ".join(cur + [s])) > max_tokens
        if shift or too_big:
            groups.append(" ".join(cur)); cur = [s]
        else:
            cur.append(s)
        prev_set = cur_set
    if cur: groups.append(" ".join(cur))
    return groups

def biomedical_chunks(text, max_tokens=256):
    # Keep relation/negation-bearing sentences attached to their neighbour when budget allows.
    sents = sentences(text)
    if not sents:
        return []
    groups, cur, cur_len = [], [], 0
    for s in sents:
        s_len = n_tokens(s)
        relationish = bool(BIO["relation"].search(s) or BIO["negation"].search(s))
        exceeds = bool(cur and cur_len + s_len > max_tokens)
        if exceeds and not (relationish and cur_len < int(max_tokens * 1.15)):
            groups.append(" ".join(cur)); cur, cur_len = [s], s_len
        else:
            cur.append(s); cur_len += s_len
    if cur: groups.append(" ".join(cur))
    return groups

TOOLBOX = {
    "fixed": fixed_chunks,
    "semantic": semantic_chunks,
    "biomedical": biomedical_chunks,
    "recursive": recursive_chunks,
    "proposition": proposition_chunks,
    "parent_child": parent_child_chunks,
    "late": late_spans,
}
print("Tools:", list(TOOLBOX))

Tools: ['fixed', 'semantic', 'biomedical', 'recursive', 'proposition', 'parent_child', 'late']


## 3. Document profiler

The router needs inexpensive signals:

- length
- sentence count
- sentence-length variability
- entity density
- relation density
- negation density
- lexical shift between adjacent sentences

These are **routing features**, not final semantic truth.

In [5]:
def document_profile(text):
    s = sentences(text)
    t = tokens(text)
    if not s:
        return {"tokens":0,"sentences":0,"avg_sentence_tokens":0,"sentence_std":0,
                "entity_density":0,"relation_density":0,"negation_density":0,"lexical_shift":0,"semantic_variance":0,"cross_sentence_reference_density":0,"section_density":0}

    lens = [len(tokens(x)) for x in s]
    entity_hits = len(BIO["entity"].findall(text))
    relation_hits = len(BIO["relation"].findall(text))
    negation_hits = len(BIO["negation"].findall(text))

    # Back-reference cues (pronouns / "this treatment"-style anaphora) that
    # signal a sentence depends on earlier context (Reference Completeness risk).
    REF_CUE = re.compile(r"\\b(?:this|that|these|those|it|they|them|such|the former|the latter|herein|aforementioned)\\b", re.I)
    ref_sentences = sum(1 for x in s if REF_CUE.search(x))
    # Section markers (headings, "Methods:", "Results:", bullet/numbered leads).
    SECTION_CUE = re.compile(r"(?m)^(?:\\s*[-*\\u2022]|\\s*\\d+[.)]|\\s*[A-Z][A-Za-z ]{2,20}:)")
    section_hits = len(SECTION_CUE.findall(text))

    shifts = []
    for a,b in zip(s[:-1], s[1:]):
        A = set(re.findall(r"[A-Za-z0-9-]+", a.lower()))
        B = set(re.findall(r"[A-Za-z0-9-]+", b.lower()))
        shifts.append(1 - len(A & B) / max(1, len(A | B)))

    return {
        "tokens": len(t),
        "sentences": len(s),
        "avg_sentence_tokens": float(np.mean(lens)),
        "sentence_std": float(np.std(lens)),
        "entity_density": entity_hits / max(1, len(t)),
        "relation_density": relation_hits / max(1, len(t)),
        "negation_density": negation_hits / max(1, len(t)),
        "lexical_shift": float(np.mean(shifts)) if shifts else 0.0,
        "semantic_variance": float(np.mean(shifts)) if shifts else 0.0,
        "cross_sentence_reference_density": ref_sentences / max(1, len(s)),
        "section_density": section_hits / max(1, len(s)),
    }

profile = document_profile(demo)
profile

{'tokens': 18,
 'sentences': 3,
 'avg_sentence_tokens': 6.0,
 'sentence_std': 0.816496580927726,
 'entity_density': 0.1111111111111111,
 'relation_density': 0.05555555555555555,
 'negation_density': 0.0,
 'lexical_shift': 0.9128787878787878,
 'semantic_variance': 0.9128787878787878,
 'cross_sentence_reference_density': 0.0,
 'section_density': 0.0}

## 4. Adaptive router — cheap/default path

Initial policy:

- short document → recursive
- high biomedical signal → biomedical
- high topic shift → semantic
- long document → parent-child
- otherwise → recursive

This is deliberately transparent and deterministic.

Later, measured retrieval outcomes will teach a learned routing policy.

In [6]:
def route_rule_based(p):
    if p["tokens"] <= 180:
        return "recursive", 0.90, "short document"

    if p["relation_density"] >= 0.015 and p["negation_density"] > 0:
        return "biomedical", 0.82, "relation + negation density"

    if p["entity_density"] >= 0.012:
        return "biomedical", 0.78, "high biomedical entity density"

    if p["lexical_shift"] >= 0.65:
        return "semantic", 0.76, "high estimated topic shift"

    if p["tokens"] >= 700:
        return "parent_child", 0.73, "long document"

    return "recursive", 0.65, "default structure-preserving route"

route_rule_based(profile)

('recursive', 0.9, 'short document')

## 5. Candidate evaluator

The router needs to compare alternatives.

We calculate an **intrinsic score** from:
- boundary quality
- size compliance
- biomedical signal preservation
- redundancy

This is only a pre-retrieval diagnostic.

The real oracle will later come from downstream retrieval utility.

In [7]:
def boundary_quality(chunks):
    if not chunks: return 0.0
    return np.mean([bool(re.search(r"[.!?\)\]]$", c.strip())) for c in chunks])

def biomedical_signal_score(chunks):
    if not chunks: return 0.0
    total = 0.0
    for c in chunks:
        total += int(bool(BIO["entity"].search(c)))
        total += 0.5 * int(bool(BIO["relation"].search(c)))
        total += 0.5 * int(bool(BIO["negation"].search(c)))
    return min(1.0, total / max(1, len(chunks)*2))

def intrinsic_metrics(source, chunks, max_tokens=384):
    return {
        "chunk_count": len(chunks),
        "boundary_quality": float(boundary_quality(chunks)),
        "size_compliance": float(np.mean([n_tokens(c) <= max_tokens for c in chunks])) if chunks else 0,
        "biomedical_signal": float(biomedical_signal_score(chunks)),
        "redundancy": float(pd.Series(chunks).duplicated().mean()) if chunks else 0,
        "token_expansion": sum(n_tokens(c) for c in chunks) / max(1, n_tokens(source)),
    }

def intrinsic_score(m):
    return (
        0.30*m["boundary_quality"]
        + 0.25*m["size_compliance"]
        + 0.25*m["biomedical_signal"]
        + 0.20*(1-min(1,m["redundancy"]))
    )

rows = []
for name, fn in TOOLBOX.items():
    ch = fn(demo)
    m = intrinsic_metrics(demo, ch)
    m["strategy"] = name
    m["intrinsic_score"] = intrinsic_score(m)
    rows.append(m)

pd.DataFrame(rows).sort_values("intrinsic_score", ascending=False)

,chunk_count,boundary_quality,size_compliance,biomedical_signal,redundancy,token_expansion,strategy,intrinsic_score
0,1,1.0,1.0,0.750000,0.0,1.0,fixed,0.937500
2,1,1.0,1.0,0.750000,0.0,1.0,biomedical,0.937500
3,1,1.0,1.0,0.750000,0.0,1.0,recursive,0.937500
5,1,1.0,1.0,0.750000,0.0,1.0,parent_child,0.937500
6,1,1.0,1.0,0.750000,0.0,1.0,late,0.937500
1,3,1.0,1.0,0.416667,0.0,1.0,semantic,0.854167
4,3,1.0,1.0,0.416667,0.0,1.0,proposition,0.854167


## 6. Bounded agentic chunking

The agent loop is:

```text
PROFILE → PROPOSE → EXECUTE → EVALUATE → STOP
                              ↘ retry with another tool
```

Safety constraints:
- closed tool set
- maximum attempts
- deterministic validation
- no arbitrary code execution

In [8]:
@dataclass
class AgentDecision:
    parent_passage_id: str
    selected_strategy: str
    confidence: float
    reason: str
    attempts: int
    stopped_by: str
    candidate_scores: dict

def run_agent(parent_id, text, max_attempts=3):
    p = document_profile(text)
    proposed, conf, reason = route_rule_based(p)

    candidate_scores = {}
    candidate_chunks = {}

    for name, fn in TOOLBOX.items():
        ch = fn(text)
        m = intrinsic_metrics(text, ch)
        m["intrinsic_score"] = intrinsic_score(m)
        candidate_scores[name] = m
        candidate_chunks[name] = ch

    if proposed not in candidate_scores:
        proposed = "recursive"

    if conf >= 0.70:
        selected = proposed
        stopped_by = "confident_rule_route"
        attempts = 1
    else:
        selected = max(candidate_scores, key=lambda x: candidate_scores[x]["intrinsic_score"])
        stopped_by = "intrinsic_candidate_comparison"
        attempts = min(max_attempts, len(candidate_scores))

    return AgentDecision(
        parent_passage_id=parent_id,
        selected_strategy=selected,
        confidence=float(conf),
        reason=reason,
        attempts=attempts,
        stopped_by=stopped_by,
        candidate_scores=candidate_scores,
    )

decision = run_agent("DEMO", demo)
print(json.dumps(asdict(decision), indent=2, default=str))

{
  "parent_passage_id": "DEMO",
  "selected_strategy": "recursive",
  "confidence": 0.9,
  "reason": "short document",
  "attempts": 1,
  "stopped_by": "confident_rule_route",
  "candidate_scores": {
    "fixed": {
      "chunk_count": 1,
      "boundary_quality": 1.0,
      "size_compliance": 1.0,
      "biomedical_signal": 0.75,
      "redundancy": 0.0,
      "token_expansion": 1.0,
      "intrinsic_score": 0.9375
    },
    "semantic": {
      "chunk_count": 3,
      "boundary_quality": 1.0,
      "size_compliance": 1.0,
      "biomedical_signal": 0.4166666666666667,
      "redundancy": 0.0,
      "token_expansion": 1.0,
      "intrinsic_score": 0.8541666666666667
    },
    "biomedical": {
      "chunk_count": 1,
      "boundary_quality": 1.0,
      "size_compliance": 1.0,
      "biomedical_signal": 0.75,
      "redundancy": 0.0,
      "token_expansion": 1.0,
      "intrinsic_score": 0.9375
    },
    "recursive": {
      "chunk_count": 1,
      "boundary_quality": 1.0,
      "siz

## 7. Where LLM-guided chunking enters

We do **not** force an LLM call into every document.

Trigger the expensive path when:

- router confidence < threshold
- candidate strategies disagree strongly
- document has unusual structure
- biomedical relation density is high
- later retrieval indicates the representation is failing

The LLM then chooses from the closed strategy set or proposes refined boundaries for a selected strategy.

This gives BioRAG-X a **cost-aware agentic chunking architecture**.

In [9]:
def should_trigger_agentic(profile, router_confidence, candidate_disagreement=0.0):
    if router_confidence < 0.70:
        return True, "low_router_confidence"
    if profile["lexical_shift"] > 0.75:
        return True, "high_topic_shift"
    if profile["relation_density"] > 0.02 and profile["sentences"] > 5:
        return True, "high_relation_density"
    if candidate_disagreement > 0.25:
        return True, "candidate_disagreement"
    return False, "normal_route"

print(should_trigger_agentic(profile, 0.65, 0.10))

(True, 'low_router_confidence')


## 7b. Agentic run with a REAL LLM-guided path

This ties everything together. For a document we:

1. profile it and get the deterministic route + confidence,
2. compute candidate disagreement across the toolbox,
3. ask `should_trigger_agentic(...)` whether the expensive path is warranted,
4. **only then** call the Azure LLM to propose boundaries (validated), otherwise
   use the deterministic selection.

Every decision records whether the LLM was actually used, the validation result,
and the fallback reason — so the cost of the LLM path is auditable.


In [10]:
def candidate_disagreement(text):
    """1 - (share of the single most-common chunk count across tools). Cheap proxy."""
    counts = []
    for name, fn in TOOLBOX.items():
        try:
            counts.append(len(fn(text)))
        except Exception:
            pass
    if not counts:
        return 0.0
    from collections import Counter as _C
    top = _C(counts).most_common(1)[0][1]
    return 1 - top / len(counts)

def run_agentic_document(parent_id, text, max_tokens=256, allow_llm=True):
    p = document_profile(text)
    route, conf, reason = route_rule_based(p)
    if route not in TOOLBOX:
        route = "recursive"
    disagree = candidate_disagreement(text)
    trigger, trigger_reason = should_trigger_agentic(p, conf, disagree)

    rec = {
        "parent_passage_id": parent_id,
        "deterministic_route": route,
        "router_confidence": float(conf),
        "router_reason": reason,
        "candidate_disagreement": float(disagree),
        "agentic_triggered": bool(trigger),
        "trigger_reason": trigger_reason,
        "llm_used": False,
        "llm_valid": False,
        "fallback": None,
        "final_strategy": route,
        "n_chunks": None,
    }

    if trigger and llm_available and allow_llm:
        chunks, meta = llm_guided_chunks(text, max_tokens)
        rec["llm_used"] = meta["llm_used"]
        rec["llm_valid"] = meta["valid"]
        rec["fallback"] = meta["fallback"]
        rec["final_strategy"] = "llm_guided" if meta["valid"] else route
        if not meta["valid"]:
            chunks = TOOLBOX[route](text)
    else:
        chunks = TOOLBOX[route](text)
        if trigger and not llm_available:
            rec["fallback"] = "llm_unavailable"
        elif trigger and not allow_llm:
            rec["fallback"] = "llm_budget_exhausted"

    rec["n_chunks"] = len(chunks)
    return rec, chunks

# Demo on the running example.
_demo_rec, _demo_chunks = run_agentic_document("DEMO", demo)
print(json.dumps(_demo_rec, indent=2))
print("chunks:", _demo_chunks)
print("cumulative llm usage:", llm_usage)


{
  "parent_passage_id": "DEMO",
  "deterministic_route": "recursive",
  "router_confidence": 0.9,
  "router_reason": "short document",
  "candidate_disagreement": 0.2857142857142857,
  "agentic_triggered": true,
  "trigger_reason": "high_topic_shift",
  "llm_used": true,
  "llm_valid": true,
  "fallback": null,
  "final_strategy": "llm_guided",
  "n_chunks": 3
}
chunks: ['BRCA1 participates in DNA repair.', 'Loss of BRCA1 causes homologous recombination deficiency.', 'PARP inhibitors can exploit this deficiency.']
cumulative llm usage: {'calls': 1, 'prompt_tokens': 173, 'completion_tokens': 69, 'total_tokens': 242, 'seconds': 2.296189900007448}


## 8. Chunk-selection accuracy and regret

These are the key metrics for adaptive/agentic chunking.

Suppose true measured downstream utility is:

```text
recursive     0.72
semantic      0.81
biomedical    0.84
proposition   0.76
parent-child  0.79
```

If the router chooses `recursive`:

```text
selection_accuracy = 0
regret = 0.84 - 0.72 = 0.12
```

If it chooses `biomedical`:

```text
selection_accuracy = 1
regret = 0
```

The **real** utility scores will come from later retrieval experiments; this notebook only establishes the metric interface.

In [11]:
def selection_metrics(selected_strategy, strategy_utilities):
    if not strategy_utilities:
        return {"oracle_strategy":None, "accuracy":np.nan, "regret":np.nan}

    oracle = max(strategy_utilities, key=strategy_utilities.get)
    oracle_score = float(strategy_utilities[oracle])
    selected_score = float(strategy_utilities.get(selected_strategy, 0.0))

    return {
        "oracle_strategy": oracle,
        "accuracy": float(selected_strategy == oracle),
        "regret": oracle_score - selected_score,
        "oracle_score": oracle_score,
        "selected_score": selected_score,
    }

selection_metrics("recursive", {
    "recursive":0.72, "semantic":0.81,
    "biomedical":0.84, "proposition":0.76, "parent_child":0.79
})

{'oracle_strategy': 'biomedical',
 'accuracy': 0.0,
 'regret': 0.12,
 'oracle_score': 0.84,
 'selected_score': 0.72}

## 9. Router rollout over a development sample

This is a **development profile**, not the locked 100-query benchmark.

We record every routing decision so later retrieval results can be joined back to:
- document characteristics
- selected strategy
- router confidence
- candidate strategy outcomes

In [12]:
ROUTER_SAMPLE_N = min(2000, len(passages))
sample = passages.sample(n=ROUTER_SAMPLE_N, random_state=SEED)

# The deterministic router runs over ALL sampled docs (fast). The expensive
# LLM path is capped by a budget so the rollout stays cost-aware, exactly as the
# research design intends (fast default; LLM only selectively). Documents that
# WOULD trigger the LLM beyond the budget are recorded as agentic_triggered=True
# with fallback='llm_budget_exhausted', so the telemetry stays honest.
LLM_ROLLOUT_BUDGET = 60
_llm_spent = 0

decision_rows = []
for row in sample[["canonical_passage_id", "retrieval_text"]].itertuples(index=False):
    allow = _llm_spent < LLM_ROLLOUT_BUDGET
    rec, _chunks = run_agentic_document(row.canonical_passage_id, row.retrieval_text, allow_llm=allow)
    if rec["llm_used"]:
        _llm_spent += 1
    p = document_profile(row.retrieval_text)
    decision_rows.append({**rec, **p})

router_decisions = pd.DataFrame(decision_rows)
print("Rollout rows:", len(router_decisions))
print("Agentic triggered:", int(router_decisions["agentic_triggered"].sum()),
      f"({router_decisions['agentic_triggered'].mean():.1%})")
print("LLM used:", int(router_decisions["llm_used"].sum()),
      "| LLM valid:", int(router_decisions["llm_valid"].sum()),
      f"| budget={LLM_ROLLOUT_BUDGET}")
print("Fallback reasons:")
print(router_decisions["fallback"].value_counts(dropna=False).to_string())
print("Final strategy distribution:")
print(router_decisions["final_strategy"].value_counts().to_string())
display(router_decisions.head())


Rollout rows: 2000
Agentic triggered: 1385 (69.2%)
LLM used: 60 | LLM valid: 60 | budget=60
Fallback reasons:
fallback
llm_budget_exhausted    1325
None                     675
Final strategy distribution:
final_strategy
recursive       1115
semantic         780
llm_guided        60
biomedical        43
parent_child       2


,parent_passage_id,deterministic_route,router_confidence,router_reason,candidate_disagreement,agentic_triggered,trigger_reason,llm_used,llm_valid,fallback,...,sentences,avg_sentence_tokens,sentence_std,entity_density,relation_density,negation_density,lexical_shift,semantic_variance,cross_sentence_reference_density,section_density
0,BIOP-29449511,recursive,0.9,short document,0.428571,True,high_topic_shift,True,True,None,...,6,20.833333,2.544056,0.0,0.0,0.0,0.919990,0.919990,0.0,0.0
1,BIOP-25542770,recursive,0.9,short document,0.000000,False,normal_route,False,False,None,...,0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
2,BIOP-23180957,recursive,0.9,short document,0.000000,False,normal_route,False,False,None,...,0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
3,BIOP-15157997,recursive,0.9,short document,0.428571,True,high_topic_shift,True,True,None,...,7,23.142857,10.561772,0.0,0.0,0.0,0.808032,0.808032,0.0,0.0
4,BIOP-2724231,recursive,0.9,short document,0.000000,False,normal_route,False,False,None,...,0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0


In [13]:
distribution = (
    router_decisions["final_strategy"]
    .value_counts()
    .rename_axis("strategy")
    .reset_index(name="documents")
)
distribution["pct"] = 100*distribution["documents"]/len(router_decisions)
display(distribution)

print(
    "Low-confidence rate:",
    f"{(router_decisions['router_confidence'] < 0.70).mean():.2%}"
)

,strategy,documents,pct
0,recursive,1115,55.75
1,semantic,780,39.00
2,llm_guided,60,3.00
3,biomedical,43,2.15
4,parent_child,2,0.10


Low-confidence rate: 0.20%


## 9b. Chunk-quality scoring — how good ARE these chunkings?

Producing chunks is not enough; we must be able to **verify their quality**. We
score every strategy (including the real LLM-guided output) with concrete,
retriever-free metrics reused from the Notebook 04 discipline:

- **entity / relation / negation integrity** — does each biomedical signal in the
  source survive intact inside a single chunk (not split across a boundary)?
- **boundary quality** — fraction of chunks that end on a sentence/clause boundary.
- **size compliance** — fraction within the token budget.
- **redundancy** — duplicate-chunk rate.

We then run the real LLM-guided path on a small sample of triggered documents and
compare its quality against the deterministic strategies on the *same* documents.
This is intrinsic quality, not the final retrieval verdict (deferred to NB06/07).


In [14]:
# Named adaptive-chunking metrics from the research doc (§10): ICC / DCC / RC.
_RC_CUE = re.compile(r"^\s*(?:this|that|these|those|it|they|them|such|the former|the latter)\b", re.I)

def _tokset(x):
    return set(re.findall(r"[A-Za-z0-9-]+", x.lower()))

def adaptive_metrics(source_text, chunks):
    """ICC = mean intra-chunk adjacent-sentence cohesion;
       DCC = mean overlap of each chunk's vocabulary with the whole document;
       RC  = fraction of chunks that do NOT start with a dangling back-reference."""
    if not chunks:
        return {"ICC": 0.0, "DCC": 0.0, "RC": 0.0}
    doc_set = _tokset(source_text)

    # ICC: within each chunk, average adjacent-sentence token overlap.
    icc_vals = []
    for c in chunks:
        cs = sentences(c)
        if len(cs) < 2:
            icc_vals.append(1.0)  # single-sentence chunk is trivially cohesive
            continue
        ov = []
        for a, b in zip(cs[:-1], cs[1:]):
            A, B = _tokset(a), _tokset(b)
            ov.append(len(A & B) / max(1, len(A | B)))
        icc_vals.append(float(np.mean(ov)) if ov else 1.0)

    # DCC: how much of each chunk's vocabulary is grounded in the document.
    dcc_vals = []
    for c in chunks:
        cset = _tokset(c)
        dcc_vals.append(len(cset & doc_set) / max(1, len(cset)))

    # RC: chunk should not begin with an unresolved reference.
    rc_ok = sum(0 if _RC_CUE.search(c.strip()) else 1 for c in chunks)

    return {
        "ICC": float(np.mean(icc_vals)),
        "DCC": float(np.mean(dcc_vals)),
        "RC": rc_ok / len(chunks),
    }

# Reusable, source-vs-chunk integrity + intrinsic quality for a SINGLE document.
def doc_integrity(source_text, chunks):
    ent_tot = ent_ok = rel_tot = rel_ok = neg_tot = neg_ok = 0
    ents = BIO["entity"].findall(source_text)
    for e in ents:
        ent_tot += 1
        if any(e.lower() in c.lower() for c in chunks):
            ent_ok += 1
    for neg in BIO["negation"].findall(source_text):
        neg_tot += 1
        if any(neg.lower() in c.lower() for c in chunks):
            neg_ok += 1
    for sent in sentences(source_text):
        if BIO["relation"].search(sent):
            rel_tot += 1
            norm = re.sub(r"\s+", " ", sent).strip().lower()
            if any(norm in re.sub(r"\s+", " ", c).strip().lower() for c in chunks):
                rel_ok += 1
    return ent_ok, ent_tot, rel_ok, rel_tot, neg_ok, neg_tot

def quality_row(name, source_text, chunks, max_tokens=256):
    m = intrinsic_metrics(source_text, chunks, max_tokens)
    eo, et, ro, rt, no, nt = doc_integrity(source_text, chunks)
    am = adaptive_metrics(source_text, chunks)
    return {
        "strategy": name,
        "n_chunks": len(chunks),
        "boundary_quality": m["boundary_quality"],
        "size_compliance": m["size_compliance"],
        "redundancy": m["redundancy"],
        "intrinsic_score": intrinsic_score(m),
        "ICC": am["ICC"], "DCC": am["DCC"], "RC": am["RC"],
        "_eo": eo, "_et": et, "_ro": ro, "_rt": rt, "_no": no, "_nt": nt,
    }

# Sample documents that the agentic trigger fires on, so the LLM path is exercised.
QUALITY_SAMPLE_N = min(40, len(passages))
_qsample = passages.sample(n=min(400, len(passages)), random_state=SEED)

_triggered = []
for row in _qsample[["canonical_passage_id", "retrieval_text"]].itertuples(index=False):
    p = document_profile(row.retrieval_text)
    route, conf, _ = route_rule_based(p)
    if route not in TOOLBOX:
        route = "recursive"
    trig, _ = should_trigger_agentic(p, conf, candidate_disagreement(row.retrieval_text))
    if trig:
        _triggered.append((row.canonical_passage_id, row.retrieval_text))
    if len(_triggered) >= QUALITY_SAMPLE_N:
        break

print(f"Scoring {len(_triggered)} agentic-triggered documents "
      f"(llm_available={llm_available}).")

_agg = {}   # strategy -> list of rows
_int = {}   # strategy -> [eo,et,ro,rt,no,nt] accumulators

def _accum(strategy, r):
    _agg.setdefault(strategy, []).append(r)
    a = _int.setdefault(strategy, [0, 0, 0, 0, 0, 0])
    a[0] += r["_eo"]; a[1] += r["_et"]; a[2] += r["_ro"]; a[3] += r["_rt"]; a[4] += r["_no"]; a[5] += r["_nt"]

for pid, text in _triggered:
    # Deterministic strategies
    for name in ["recursive", "semantic", "biomedical", "proposition", "parent_child"]:
        _accum(name, quality_row(name, text, TOOLBOX[name](text)))
    # Real LLM-guided
    ch, meta = llm_guided_chunks(text)
    label = "llm_guided" if meta["valid"] else "llm_guided(fallback)"
    _accum(label, quality_row(label, text, ch))

rows = []
for strategy, items in _agg.items():
    df = pd.DataFrame(items)
    eo, et, ro, rt, no, nt = _int[strategy]
    rows.append({
        "strategy": strategy,
        "docs": len(items),
        "mean_n_chunks": round(df["n_chunks"].mean(), 2),
        "boundary_quality": round(df["boundary_quality"].mean(), 3),
        "size_compliance": round(df["size_compliance"].mean(), 3),
        "redundancy": round(df["redundancy"].mean(), 3),
        "entity_integrity": round(eo / max(1, et), 3),
        "relation_integrity": round(ro / max(1, rt), 3),
        "negation_integrity": round(no / max(1, nt), 3),
        "intrinsic_score": round(df["intrinsic_score"].mean(), 3),
        "ICC": round(df["ICC"].mean(), 3),
        "DCC": round(df["DCC"].mean(), 3),
        "RC": round(df["RC"].mean(), 3),
    })

chunk_quality = pd.DataFrame(rows).sort_values("intrinsic_score", ascending=False).reset_index(drop=True)
display(chunk_quality)
chunk_quality.to_csv(ARTIFACT_DIR / "chunk_quality_scores.csv", index=False)
print("\nLLM usage after scoring:", llm_usage)


Scoring 40 agentic-triggered documents (llm_available=True).


,strategy,docs,mean_n_chunks,boundary_quality,size_compliance,redundancy,entity_integrity,relation_integrity,negation_integrity,intrinsic_score,ICC,DCC,RC
0,recursive,40,1.18,1.000,1.000,0.000,1.0,1.000,1.0,0.819,0.167,1.0,0.962
1,biomedical,40,1.10,1.000,0.912,0.000,1.0,1.000,1.0,0.798,0.134,1.0,0.975
2,llm_guided,40,4.75,1.000,1.000,0.000,1.0,1.000,1.0,0.774,0.566,1.0,0.942
3,semantic,40,6.72,1.000,1.000,0.000,1.0,1.000,1.0,0.771,0.787,1.0,0.938
4,proposition,40,18.05,0.544,1.000,0.005,1.0,0.212,1.0,0.621,1.000,1.0,0.973
5,parent_child,40,2.80,0.421,1.000,0.000,1.0,0.909,1.0,0.620,0.152,1.0,0.971



LLM usage after scoring: {'calls': 196, 'prompt_tokens': 92438, 'completion_tokens': 17827, 'total_tokens': 110265, 'seconds': 501.4433988999808}


## 9c. Selection accuracy and regret on a REAL intrinsic oracle

Section 8 defined the accuracy/regret interface on hypothetical numbers. Now that
9b produced a real per-strategy intrinsic-quality table, we can measure the
router for real: for each triggered document we compute every deterministic
strategy's intrinsic score, treat the best as the **oracle**, and compare it with
what the router actually chose.

This is an **intrinsic** oracle (pre-retrieval). The definitive
downstream-utility regret still comes from Notebook 06/11; this gives an honest
early read on routing quality using measured chunk quality rather than guesses.


In [15]:
# Per-document intrinsic scores for each deterministic strategy, then compare
# the router's choice against the intrinsic-best (oracle) on the triggered sample.
DET_STRATEGIES = ["recursive", "semantic", "biomedical", "proposition", "parent_child"]

sel_rows = []
for pid, text in _triggered:
    utils = {}
    for name in DET_STRATEGIES:
        ch = TOOLBOX[name](text)
        utils[name] = intrinsic_score(intrinsic_metrics(text, ch))
    p = document_profile(text)
    route, conf, _ = route_rule_based(p)
    if route not in TOOLBOX:
        route = "recursive"
    sm = selection_metrics(route, utils)
    sel_rows.append({
        "parent_passage_id": pid,
        "router_choice": route,
        "oracle_strategy": sm["oracle_strategy"],
        "accuracy": sm["accuracy"],
        "regret": sm["regret"],
    })

selection_eval = pd.DataFrame(sel_rows)
if len(selection_eval):
    acc = float(selection_eval["accuracy"].mean())
    reg = float(selection_eval["regret"].mean())
    print(f"Documents evaluated: {len(selection_eval)}")
    print(f"Router selection accuracy (intrinsic oracle): {acc:.3f}")
    print(f"Mean routing regret (intrinsic oracle):       {reg:.4f}")
    print("\nOracle strategy distribution:")
    print(selection_eval["oracle_strategy"].value_counts().to_string())
    selection_eval.to_csv(ARTIFACT_DIR / "selection_accuracy_regret.csv", index=False)
    selection_summary = {"documents": int(len(selection_eval)),
                         "selection_accuracy_intrinsic": round(acc, 4),
                         "mean_regret_intrinsic": round(reg, 4)}
else:
    selection_summary = {"documents": 0}
    print("No triggered documents to evaluate.")
selection_summary


Documents evaluated: 40
Router selection accuracy (intrinsic oracle): 0.325
Mean routing regret (intrinsic oracle):       0.0343

Oracle strategy distribution:
oracle_strategy
recursive     38
biomedical     2


{'documents': 40,
 'selection_accuracy_intrinsic': 0.325,
 'mean_regret_intrinsic': 0.0343}

## 10. Future learned routing

Do NOT train a router on its own decisions.

The future supervised dataset must look like:

```text
(document/query features)
          ↓
measured utility of EACH strategy
          ↓
oracle best strategy
          ↓
train/predict routing policy
```

Example:

```text
features → [
  semantic_variance,
  entity_density,
  relation_density,
  question_type,
  query lexical overlap,
  document length
]

labels → {
  fixed: 0.71,
  semantic: 0.79,
  biomedical: 0.85,
  proposition: 0.75,
  parent_child: 0.82
}
```

The model then learns which representation is likely to maximize downstream utility.

## 11. Expected utility for production routing

Long-term utility should include both quality and cost:

```text
Utility =
  retrieval quality
+ evidence completeness
+ answer quality
+ grounding
- latency
- index expansion
- context cost
```

This prevents a chunker from "winning" simply because it generates enormous numbers of highly localized chunks.

We will populate true utility values after Notebook 06/11 provides downstream retrieval and answer results.

In [16]:
feature_cols = [
    "tokens","sentences","avg_sentence_tokens","sentence_std",
    "entity_density","relation_density","negation_density","lexical_shift"
]

router_training_shell = router_decisions[
    ["parent_passage_id","final_strategy","router_confidence"] + feature_cols
].copy()

print("Training shell created:", router_training_shell.shape)
print("Important: this is NOT supervised ground truth yet.")

Training shell created: (2000, 11)
Important: this is NOT supervised ground truth yet.


## 12. Persist artifacts

These outputs are inputs to the later evaluation and routing-policy notebooks.

In [17]:
router_path = ARTIFACT_DIR / "router_decisions.parquet"
router_decisions.to_parquet(router_path, index=False)

manifest = {
    "notebook": "05_agentic_chunking",
    "seed": SEED,
    "router_sample_size": int(ROUTER_SAMPLE_N),
    "llm_rollout_budget": int(LLM_ROLLOUT_BUDGET),
    "strategy_space": list(TOOLBOX.keys()),
    "llm_guided": {
        "enabled": bool(llm_available),
        "backend": LLM_MODEL_NAME,
        "usage": llm_usage,
        "proposal_only_validated": True,
    },
    "quality_report": str(ARTIFACT_DIR / "chunk_quality_scores.csv"),
    "quality_metrics": ["boundary_quality", "size_compliance", "redundancy",
                        "entity_integrity", "relation_integrity", "negation_integrity",
                        "ICC", "DCC", "RC", "intrinsic_score"],
    "profile_fields": ["tokens", "sentences", "avg_sentence_tokens", "sentence_std",
                       "entity_density", "relation_density", "negation_density",
                       "lexical_shift", "semantic_variance",
                       "cross_sentence_reference_density", "section_density"],
    "selection_eval_intrinsic": selection_summary,
    "selection_regret_report": str(ARTIFACT_DIR / "selection_accuracy_regret.csv"),
    "agentic_policy": {
        "max_attempts": 3,
        "closed_tool_space": True,
        "llm_guided": True,
        "validator_required": True
    },
    "metrics": [
        "chunk_selection_accuracy",
        "chunking_regret",
        "selected_strategy_utility",
        "oracle_strategy_utility"
    ],
    "status": {
        "downstream_utility_available": False,
        "router_is_provisional": True
    }
}

manifest_path = ARTIFACT_DIR / "agentic_chunking_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(router_path)
print(manifest_path)

artifacts\05_agentic_chunking\router_decisions.parquet
artifacts\05_agentic_chunking\agentic_chunking_manifest.json


# 13. What we learned

### LLM-guided chunking
Useful for nuanced boundary proposals, but must be structured and validated.

### Agentic chunking
Adds iterative strategy selection/refinement, so it should be bounded and selectively invoked.

### Adaptive routing
Provides the cheap production default.

### Chunk-selection accuracy
Did we choose the strategy that actually performs best?

### Chunking regret
How far were we from the best strategy?

### Core research principle

> **The agent itself is not the result. The measured improvement in downstream retrieval/answer quality per unit of additional cost is the result.**

---

## Handoff

**Notebook 06 — `06_dense_embeddings_and_ann`**

will connect these representations to:

- MedCPT
- exact dense retrieval
- ANN / HNSW
- IVF/IVF-PQ experiments
- real late-chunk embeddings
- Recall@K / MRR / nDCG
- evidence recall
- ANN recall loss / speedup

Those results will provide the real strategy-utility labels needed to train and validate the adaptive chunking router.